In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/Chattogram_Landmark_Dataset"

print("Project exists:", os.path.exists(BASE_DIR))

print("\nProject folders:")

for item in os.listdir(BASE_DIR):
    print(" -", item)

Project exists: True

Project folders:
 - 01_inventory
 - 02_wikimedia_scan
 - 03_candidate_pool
 - 04_final_dataset
 - 05_reports
 - 02_availability
 - 03_candidates
 - 04_review
 - 05_final_dataset
 - 06_reports
 - 07_cache


In [ ]:
import os

for root, dirs, files in os.walk(BASE_DIR):

    image_count = sum(
        f.lower().endswith(
            (".jpg", ".jpeg", ".png", ".webp")
        )
        for f in files
    )

    if image_count > 0:
        print(
            f"{root}  -->  {image_count} images"
        )

/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/Patenga_Beach  -->  119 images
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/Foys_Lake  -->  85 images
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/Parki_Beach  -->  13 images
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/Chandranath_Hill  -->  101 images
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/Chandranath_Temple  -->  96 images
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/Sitakunda_Eco_Park  -->  114 images
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/Guliakhali_Beach  -->  88 images
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/Banshbaria_Beach  -->  58 images
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/Kalurghat_Bridge  -->  86 images
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/Shah_Amanat_Bridge  -->  64 images
/content/drive/MyDrive/Cha

In [ ]:
import glob

csv_files = glob.glob(
    BASE_DIR + "/**/*.csv",
    recursive=True
)

print("CSV files found:\n")

for f in csv_files:
    print(f)

CSV files found:

/content/drive/MyDrive/Chattogram_Landmark_Dataset/02_availability/availability_report.csv
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/candidate_metadata.csv
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/clean_candidate_metadata.csv
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/recovered_clean_metadata.csv
/content/drive/MyDrive/Chattogram_Landmark_Dataset/04_review/duplicate_analysis/exact_duplicates.csv
/content/drive/MyDrive/Chattogram_Landmark_Dataset/04_review/duplicate_analysis/identical_phash_groups.csv
/content/drive/MyDrive/Chattogram_Landmark_Dataset/04_review/duplicate_analysis/near_duplicate_pairs.csv


In [ ]:
import pandas as pd
import os

CLEAN_METADATA = (
    "/content/drive/MyDrive/"
    "Chattogram_Landmark_Dataset/"
    "03_candidates/"
    "clean_candidate_metadata.csv"
)

clean_df = pd.read_csv(CLEAN_METADATA)

print("Clean metadata loaded successfully!")
print("Number of records:", len(clean_df))
print("Columns:")
print(clean_df.columns.tolist())

Clean metadata loaded successfully!
Number of records: 5340
Columns:
['landmark', 'district', 'filename', 'path', 'sha256', 'title', 'file_url', 'description_url', 'license', 'artist', 'width', 'height', 'status', 'phash']


In [ ]:
print("Total records:", len(clean_df))
print("Unique landmarks:", clean_df["landmark"].nunique())

print("\nMissing values:")
print(clean_df.isna().sum())

print("\nStatus distribution:")
print(clean_df["status"].value_counts(dropna=False))

print("\nMissing pHash:")
print(clean_df["phash"].isna().sum())

print("\nMissing SHA-256:")
print(clean_df["sha256"].isna().sum())

Total records: 5340
Unique landmarks: 80

Missing values:
landmark            0
district            0
filename            0
path                0
sha256              0
title               0
file_url            0
description_url     0
license             0
artist             82
width               0
height              0
status              0
phash               0
dtype: int64

Status distribution:
status
downloaded    5340
Name: count, dtype: int64

Missing pHash:
0

Missing SHA-256:
0


In [ ]:
import os

clean_df["file_exists"] = clean_df["path"].apply(os.path.exists)

print("Total records:", len(clean_df))
print("Images found:", clean_df["file_exists"].sum())
print("Images missing:", (~clean_df["file_exists"]).sum())

Total records: 5340
Images found: 5340
Images missing: 0


In [ ]:
class_counts = (
    clean_df
    .groupby(["district", "landmark"])
    .size()
    .reset_index(name="image_count")
    .sort_values("image_count", ascending=False)
)

print(class_counts.to_string(index=False))

    district                         landmark  image_count
  Coxs_Bazar               Radiant_Fish_World          120
  Chattogram                    Patenga_Beach          119
  Chattogram              Ethnological_Museum          119
     Cumilla                 Mainamati_Museum          118
  Coxs_Bazar                   Adinath_Temple          117
  Chattogram         Central_Railway_Building          116
     Cumilla                     Lalmai_Hills          116
Khagrachhari                 Mong_King_Palace          116
Brahmanbaria            Haripur_Barendra_Bari          115
  Chattogram                   Chattogram_Zoo          115
  Chattogram               Sitakunda_Eco_Park          114
  Coxs_Bazar                      Maheshkhali          113
     Cumilla                    Itakhola_Mura          112
  Chattogram       Chattogram_Railway_Station          111
     Cumilla                   Shalban_Vihara          110
   Bandarban                          Nilgiri          1

In [ ]:
targets = [100, 90, 80, 70, 60, 50, 40, 30, 20]

for target in targets:
    n_classes = (
        class_counts["image_count"] >= target
    ).sum()

    print(
        f"{target:3d} images/class -> "
        f"{n_classes:3d} classes"
    )

100 images/class ->  30 classes
 90 images/class ->  34 classes
 80 images/class ->  40 classes
 70 images/class ->  42 classes
 60 images/class ->  44 classes
 50 images/class ->  46 classes
 40 images/class ->  49 classes
 30 images/class ->  59 classes
 20 images/class ->  63 classes


In [ ]:
district_summary = (
    class_counts
    .groupby("district")
    .agg(
        number_of_classes=("landmark", "count"),
        total_images=("image_count", "sum")
    )
    .sort_values(
        "number_of_classes",
        ascending=False
    )
)

district_summary

,number_of_classes,total_images
district,,
Chattogram,23,1917
Coxs_Bazar,11,885
Bandarban,10,599
Cumilla,8,695
Rangamati,8,364
Chandpur,4,136
Brahmanbaria,4,289
Khagrachhari,4,171
Feni,3,17


In [ ]:
print(
    class_counts
    .groupby("district")["landmark"]
    .count()
)

district
Bandarban       10
Brahmanbaria     4
Chandpur         4
Chattogram      23
Coxs_Bazar      11
Cumilla          8
Feni             3
Khagrachhari     4
Lakshmipur       3
Noakhali         2
Rangamati        8
Name: landmark, dtype: int64


In [ ]:
print(
    class_counts[
        class_counts["image_count"] < 40
    ].to_string(index=False)
)

    district               landmark  image_count
  Lakshmipur          Ramgati_Beach           39
   Bandarban             Keokradong           39
Brahmanbaria         Anderson_Canal           38
  Chattogram          Cheragi_Pahar           37
  Chattogram     Chandanpura_Mosque           35
Khagrachhari           Alutila_Cave           34
    Chandpur  Chandpur_Boro_Station           34
    Noakhali     Bajra_Shahi_Mosque           32
Brahmanbaria         Arifail_Mosque           31
   Rangamati    Shuvolong_Waterfall           31
   Bandarban Meghla_Tourist_Complex           29
    Chandpur     Three_Rivers_Mouth           28
    Chandpur    Rupsha_Jamidar_Bari           21
Khagrachhari       Risang_Waterfall           20
  Coxs_Bazar         Sonadia_Island           18
  Coxs_Bazar    Kutubdia_Lighthouse           18
  Coxs_Bazar  Ramu_Buddhist_Temples           17
   Bandarban               Amiakhum           17
   Bandarban     Nafakhum_Waterfall           16
   Rangamati        

In [ ]:
print(
    clean_df["phash"]
    .dropna()
    .head(20)
)

0     d2cc2d36737c3283
1     ca28306fd37b348b
2     861877c2bdf22d19
3     85c3e2bc6c50ee93
4     fed54a9a9472829c
5     879de58ac91af454
6     aaf5b5a4cc532a4a
7     e2ecd78d94945cb0
8     d35d7cf221c85b82
9     d9dad8dbc3f29200
10    dc2baf74825bb520
11    e7c4c1f0f2f13c18
12    d568c832871d0fee
13    81a134b7926f9c7a
14    e1c58f2f92661635
15    8c1db9976b81f981
16    cd85f240607c7ab7
17    a1827d0db0fa9de1
18    d0ff01af5f20de20
19    a89517e65db80d8b
Name: phash, dtype: object


In [ ]:
print(
    "Unique pHash values:",
    clean_df["phash"].nunique()
)

print(
    "Total images:",
    len(clean_df)
)

Unique pHash values: 5132
Total images: 5340


In [ ]:
exact_duplicates = clean_df[
    clean_df.duplicated(
        subset=["sha256"],
        keep=False
    )
].sort_values("sha256")

print(
    "Records involved in exact duplicates:",
    len(exact_duplicates)
)

Records involved in exact duplicates: 398


In [ ]:
print(
    exact_duplicates[
        [
            "landmark",
            "filename",
            "sha256"
        ]
    ].head(30)
)

                             landmark  \
113                    Adinath_Temple   
3588                      Maheshkhali   
1749  Comilla_Cantonment_War_Cemetery   
3667                 Mainamati_Museum   
104                    Adinath_Temple   
3575                      Maheshkhali   
1065               Chandranath_Temple   
942                  Chandranath_Hill   
1757  Comilla_Cantonment_War_Cemetery   
3672                 Mainamati_Museum   
1820  Comilla_Cantonment_War_Cemetery   
3709                 Mainamati_Museum   
1741  Comilla_Cantonment_War_Cemetery   
3712                 Mainamati_Museum   
3589                      Maheshkhali   
71                     Adinath_Temple   
3530                      Maheshkhali   
29                     Adinath_Temple   
1776  Comilla_Cantonment_War_Cemetery   
3685                 Mainamati_Museum   
3759           Meghla_Tourist_Complex   
896                  Chandpur_Stadium   
843             Chandpur_Boro_Station   
3719            

In [ ]:
CHECKPOINT = (
    "/content/drive/MyDrive/"
    "Chattogram_Landmark_Dataset/"
    "03_candidates/"
    "recovered_clean_metadata.csv"
)

clean_df.to_csv(
    CHECKPOINT,
    index=False
)

print("Checkpoint saved:")
print(CHECKPOINT)

Checkpoint saved:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/03_candidates/recovered_clean_metadata.csv


In [ ]:
DUP_DIR = (
    "/content/drive/MyDrive/"
    "Chattogram_Landmark_Dataset/"
    "04_review/"
    "duplicate_analysis"
)

os.makedirs(
    DUP_DIR,
    exist_ok=True
)

print(DUP_DIR)

/content/drive/MyDrive/Chattogram_Landmark_Dataset/04_review/duplicate_analysis


In [ ]:
phash_df = clean_df.copy()

phash_df = phash_df[
    phash_df["phash"].notna()
].copy()

phash_df["phash"] = (
    phash_df["phash"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Records with pHash:", len(phash_df))
print("Unique pHash values:", phash_df["phash"].nunique())

Records with pHash: 5340
Unique pHash values: 5132


In [ ]:
print("pHash length distribution:")
print(
    phash_df["phash"]
    .str.len()
    .value_counts()
    .sort_index()
)

print("\nSample pHashes:")
print(
    phash_df["phash"]
    .head(10)
    .to_string(index=False)
)

pHash length distribution:
phash
16    5340
Name: count, dtype: int64

Sample pHashes:
d2cc2d36737c3283
ca28306fd37b348b
861877c2bdf22d19
85c3e2bc6c50ee93
fed54a9a9472829c
879de58ac91af454
aaf5b5a4cc532a4a
e2ecd78d94945cb0
d35d7cf221c85b82
d9dad8dbc3f29200


In [ ]:
sha_groups = (
    phash_df
    .groupby("sha256")
    .size()
    .reset_index(name="count")
)

exact_duplicate_groups = sha_groups[
    sha_groups["count"] > 1
].copy()

print("Exact duplicate groups:", len(exact_duplicate_groups))
print(
    "Images involved in exact duplicates:",
    exact_duplicate_groups["count"].sum()
)

Exact duplicate groups: 196
Images involved in exact duplicates: 398


In [ ]:
exact_duplicates_df = phash_df[
    phash_df["sha256"].isin(
        exact_duplicate_groups["sha256"]
    )
].sort_values("sha256")

exact_duplicates_path = os.path.join(
    DUP_DIR,
    "exact_duplicates.csv"
)

exact_duplicates_df.to_csv(
    exact_duplicates_path,
    index=False
)

print("Saved:")
print(exact_duplicates_path)

Saved:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/04_review/duplicate_analysis/exact_duplicates.csv


In [ ]:
phash_groups = (
    phash_df
    .groupby("phash")
    .size()
    .reset_index(name="count")
)

repeated_phash = phash_groups[
    phash_groups["count"] > 1
].copy()

print("Repeated pHash groups:", len(repeated_phash))
print(
    "Images involved:",
    repeated_phash["count"].sum()
)

Repeated pHash groups: 198
Images involved: 406


In [ ]:
identical_phash_df = phash_df[
    phash_df["phash"].isin(
        repeated_phash["phash"]
    )
].sort_values("phash")

identical_phash_path = os.path.join(
    DUP_DIR,
    "identical_phash_groups.csv"
)

identical_phash_df.to_csv(
    identical_phash_path,
    index=False
)

print("Saved:")
print(identical_phash_path)

Saved:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/04_review/duplicate_analysis/identical_phash_groups.csv


In [ ]:
def phash_to_int(phash):
    try:
        return int(phash, 16)
    except Exception:
        return None


phash_df["phash_int"] = phash_df["phash"].apply(phash_to_int)

invalid_phash = phash_df[
    phash_df["phash_int"].isna()
]

print("Invalid pHash values:", len(invalid_phash))

Invalid pHash values: 0


In [ ]:
def hamming_distance(a, b):
    return (a ^ b).bit_count()

print("Hamming function ready.")

Hamming function ready.


In [ ]:
import numpy as np

SEGMENT_BITS = 9

# Make sure pHash values are integer
phash_values = phash_df["phash_int"].astype(np.uint64).to_numpy()

segment_columns = []

for i in range(7):
    shift = i * SEGMENT_BITS
    col = f"seg_{i}"

    if shift < 64:
        phash_df[col] = (
            (phash_values >> np.uint64(shift))
            & np.uint64((1 << SEGMENT_BITS) - 1)
        ).astype(np.uint64)
    else:
        phash_df[col] = 0

    segment_columns.append(col)

print("Segment indexes created:")
print(segment_columns)

Segment indexes created:
['seg_0', 'seg_1', 'seg_2', 'seg_3', 'seg_4', 'seg_5', 'seg_6']


In [ ]:
from collections import defaultdict

MAX_DISTANCE = 6

candidate_pairs = set()

for segment in segment_columns:

    groups = phash_df.groupby(segment)

    for _, group in groups:

        indices = group.index.tolist()

        if len(indices) < 2:
            continue

        for i in range(len(indices)):
            idx1 = indices[i]
            hash1 = phash_df.at[idx1, "phash_int"]

            for j in range(i + 1, len(indices)):
                idx2 = indices[j]

                pair = (
                    min(idx1, idx2),
                    max(idx1, idx2)
                )

                candidate_pairs.add(pair)

print("Candidate pairs:", len(candidate_pairs))

Candidate pairs: 225616


In [ ]:
from tqdm.auto import tqdm
near_duplicate_pairs = []

for idx1, idx2 in tqdm(
    candidate_pairs,
    desc="Calculating Hamming distances"
):

    hash1 = phash_df.at[idx1, "phash_int"]
    hash2 = phash_df.at[idx2, "phash_int"]

    distance = hamming_distance(
        hash1,
        hash2
    )

    if distance <= MAX_DISTANCE:

        row1 = phash_df.loc[idx1]
        row2 = phash_df.loc[idx2]

        near_duplicate_pairs.append({
            "distance": distance,

            "landmark_1": row1["landmark"],
            "landmark_2": row2["landmark"],

            "district_1": row1["district"],
            "district_2": row2["district"],

            "filename_1": row1["filename"],
            "filename_2": row2["filename"],

            "path_1": row1["path"],
            "path_2": row2["path"],

            "phash_1": row1["phash"],
            "phash_2": row2["phash"]
        })

near_duplicates_df = pd.DataFrame(
    near_duplicate_pairs
)

print(
    "Near-duplicate pairs found:",
    len(near_duplicates_df)
)

Calculating Hamming distances:   0%|          | 0/225616 [00:00<?, ?it/s]

Near-duplicate pairs found: 303


In [ ]:
if len(near_duplicates_df) > 0:

    print("Hamming distance distribution:")
    print(
        near_duplicates_df[
            "distance"
        ]
        .value_counts()
        .sort_index()
    )

else:

    print("No near-duplicate pairs found.")

Hamming distance distribution:
distance
0    225
2     60
4     15
6      3
Name: count, dtype: int64


In [ ]:
cross_class = near_duplicates_df[
    near_duplicates_df["landmark_1"] !=
    near_duplicates_df["landmark_2"]
].sort_values("distance")

print(
    "Cross-class near-duplicate pairs:",
    len(cross_class)
)

print(
    cross_class.head(100).to_string(index=False)
)

Cross-class near-duplicate pairs: 303
 distance                       landmark_1             landmark_2   district_1   district_2                                                                                                                        filename_1                                                                                                                        filename_2                                                                                                                                                                                                                   path_1                                                                                                                                                                                                                path_2          phash_1          phash_2
        0                  Kutubdia_Island    Kutubdia_Lighthouse   Coxs_Bazar   Coxs_Bazar                                                      

In [ ]:
near_duplicate_path = os.path.join(
    DUP_DIR,
    "near_duplicate_pairs.csv"
)

near_duplicates_df.to_csv(
    near_duplicate_path,
    index=False
)

print("Saved:")
print(near_duplicate_path)

Saved:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/04_review/duplicate_analysis/near_duplicate_pairs.csv


In [ ]:
same_class = near_duplicates_df[
    near_duplicates_df["landmark_1"] == near_duplicates_df["landmark_2"]
].copy()

cross_class = near_duplicates_df[
    near_duplicates_df["landmark_1"] != near_duplicates_df["landmark_2"]
].copy()

print("Same-class near-duplicate pairs:", len(same_class))
print("Cross-class near-duplicate pairs:", len(cross_class))

print("\nCross-class pairs by Hamming distance:")
print(
    cross_class["distance"]
    .value_counts()
    .sort_index()
)

Same-class near-duplicate pairs: 0
Cross-class near-duplicate pairs: 303

Cross-class pairs by Hamming distance:
distance
0    225
2     60
4     15
6      3
Name: count, dtype: int64


In [ ]:
cross_pair_summary = (
    cross_class
    .groupby(
        ["landmark_1", "landmark_2"]
    )
    .agg(
        pairs=("distance", "size"),
        min_distance=("distance", "min"),
        max_distance=("distance", "max")
    )
    .reset_index()
    .sort_values(
        ["pairs", "min_distance"],
        ascending=[False, True]
    )
)

print(
    "Number of cross-class landmark pairs:",
    len(cross_pair_summary)
)

print(
    cross_pair_summary.to_string(index=False)
)

Number of cross-class landmark pairs: 129
                      landmark_1                 landmark_2  pairs  min_distance  max_distance
 Comilla_Cantonment_War_Cemetery           Mainamati_Museum     61             0             0
                  Adinath_Temple                Maheshkhali     46             0             0
                 Kutubdia_Island        Kutubdia_Lighthouse     18             0             0
                Chandranath_Hill         Chandranath_Temple     17             0             0
           Chandpur_Boro_Station         Three_Rivers_Mouth     15             0             0
                      Kaptai_Dam                Kaptai_Lake      5             0             0
                 Kutubdia_Island                Maheshkhali      4             0             4
                Chandranath_Hill           Guliakhali_Beach      3             0             0
                    Chimbuk_Hill     Meghla_Tourist_Complex      3             0             0
        

In [ ]:
problematic_classes = pd.concat([
    cross_class["landmark_1"],
    cross_class["landmark_2"]
]).value_counts()

print(
    "Classes involved in cross-class near duplicates:"
)

print(problematic_classes.to_string())

Classes involved in cross-class near duplicates:
Maheshkhali                         63
Comilla_Cantonment_War_Cemetery     61
Mainamati_Museum                    61
Adinath_Temple                      46
Kutubdia_Island                     34
Chandpur_Boro_Station               33
Chandranath_Hill                    21
Three_Rivers_Mouth                  20
Chandranath_Temple                  20
Meghla_Tourist_Complex              19
Kutubdia_Lighthouse                 18
Ramgati_Beach                       16
Kamalnagar_Coast                    15
Sonadia_Island                      15
Alutila_Cave                        14
Chandpur_Stadium                    14
Dalal_Bazar_Jamindar_Bari           14
Titas_River                         13
Wagga_Point                         13
Banshbaria_Beach                    13
Itakhola_Mura                       13
Chattogram_Zoo                       7
Kaptai_Lake                          6
Kaptai_Dam                           5
Central_Railway

In [ ]:
# Get all image paths involved in cross-class near duplicates

cross_paths = pd.unique(
    pd.concat([
        cross_class["path_1"],
        cross_class["path_2"]
    ], ignore_index=True)
)

print("Unique images involved in cross-class near duplicates:", len(cross_paths))

Unique images involved in cross-class near duplicates: 421


In [ ]:
cross_review = phash_df[
    phash_df["path"].isin(cross_paths)
].copy()

print("Images found in metadata:", len(cross_review))

cross_review[
    [
        "landmark",
        "district",
        "filename",
        "path",
        "title",
        "file_url",
        "description_url",
        "license",
        "artist",
        "width",
        "height"
    ]
].head(20)

Images found in metadata: 421


,landmark,district,filename,path,title,file_url,description_url,license,artist,width,height
0,Adinath_Temple,Coxs_Bazar,0001_File_Moheshkhali_adinath_temple01.jpg.jpg,/content/drive/MyDrive/Chattogram_Landmark_Dat...,File:Moheshkhali adinath temple01.jpg,https://upload.wikimedia.org/wikipedia/commons...,https://commons.wikimedia.org/wiki/File:Mohesh...,Public domain,"<a href=""//commons.wikimedia.org/wiki/User:Mun...",1536.0,2048.0
1,Adinath_Temple,Coxs_Bazar,0002_File_Moheshkhali_adinath_temple02.jpg.jpg,/content/drive/MyDrive/Chattogram_Landmark_Dat...,File:Moheshkhali adinath temple02.jpg,https://upload.wikimedia.org/wikipedia/commons...,https://commons.wikimedia.org/wiki/File:Mohesh...,Public domain,"<a href=""//commons.wikimedia.org/wiki/User:Mun...",1536.0,2048.0
2,Adinath_Temple,Coxs_Bazar,0003_File_Moheshkhali00005.jpg.jpg,/content/drive/MyDrive/Chattogram_Landmark_Dat...,File:Moheshkhali00005.jpg,https://upload.wikimedia.org/wikipedia/commons...,https://commons.wikimedia.org/wiki/File:Mohesh...,CC BY-SA 3.0,"<a href=""//commons.wikimedia.org/wiki/User:Bel...",2048.0,1536.0
4,Adinath_Temple,Coxs_Bazar,0005_File_Adinath_temple_1.JPG.jpg,/content/drive/MyDrive/Chattogram_Landmark_Dat...,File:Adinath temple 1.JPG,https://upload.wikimedia.org/wikipedia/commons...,https://commons.wikimedia.org/wiki/File:Adinat...,CC BY-SA 3.0,"<a href=""//commons.wikimedia.org/wiki/User:Tan...",4272.0,2848.0
5,Adinath_Temple,Coxs_Bazar,0006_File_Adinath_temple_2.JPG.jpg,/content/drive/MyDrive/Chattogram_Landmark_Dat...,File:Adinath temple 2.JPG,https://upload.wikimedia.org/wikipedia/commons...,https://commons.wikimedia.org/wiki/File:Adinat...,CC BY-SA 3.0,"<a href=""//commons.wikimedia.org/wiki/User:Tan...",2000.0,1333.0
8,Adinath_Temple,Coxs_Bazar,0009_File_Adinath_Temple_in_Moheshkhali_.jpg.jpg,/content/drive/MyDrive/Chattogram_Landmark_Dat...,File:Adinath Temple in Moheshkhali .jpg,https://upload.wikimedia.org/wikipedia/commons...,https://commons.wikimedia.org/wiki/File:Adinat...,CC BY-SA 3.0,"<a href=""//commons.wikimedia.org/wiki/User:Sal...",4160.0,3120.0
9,Adinath_Temple,Coxs_Bazar,0010_File_Adinath_Temple_Moheshkhali_.jpg.jpg,/content/drive/MyDrive/Chattogram_Landmark_Dat...,"File:Adinath Temple, Moheshkhali .jpg",https://upload.wikimedia.org/wikipedia/commons...,https://commons.wikimedia.org/wiki/File:Adinat...,CC BY-SA 3.0,"<a href=""//commons.wikimedia.org/wiki/User:Sal...",4160.0,3120.0
10,Adinath_Temple,Coxs_Bazar,0011_File_First_gate_of_Adinath_Temple_Moheshk...,/content/drive/MyDrive/Chattogram_Landmark_Dat...,"File:First gate of Adinath Temple, Moheshkhali...",https://upload.wikimedia.org/wikipedia/commons...,https://commons.wikimedia.org/wiki/File:First_...,CC BY-SA 3.0,"<a href=""//commons.wikimedia.org/wiki/User:Sal...",4160.0,3120.0
11,Adinath_Temple,Coxs_Bazar,0012_File_Adinath_Temple_premises_.jpg.jpg,/content/drive/MyDrive/Chattogram_Landmark_Dat...,File:Adinath Temple premises .jpg,https://upload.wikimedia.org/wikipedia/commons...,https://commons.wikimedia.org/wiki/File:Adinat...,CC BY-SA 3.0,"<a href=""//commons.wikimedia.org/wiki/User:Sal...",4160.0,3120.0
12,Adinath_Temple,Coxs_Bazar,0013_File_Adinath_Temple_premises_2.jpg.jpg,/content/drive/MyDrive/Chattogram_Landmark_Dat...,File:Adinath Temple premises 2.jpg,https://upload.wikimedia.org/wikipedia/commons...,https://commons.wikimedia.org/wiki/File:Adinat...,CC BY-SA 3.0,"<a href=""//commons.wikimedia.org/wiki/User:Sal...",4160.0,3120.0


In [ ]:
import re

text_cols = [
    "filename",
    "title",
    "description_url",
    "file_url"
]

review_text = (
    cross_review[text_cols]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
)

suspicious_pattern = (
    r"\.pdf|pdf|atlas|gazetteer|journal|book|"
    r"historical|report|map|document|magazine"
)

cross_review["suspicious_source"] = (
    review_text.str.contains(
        suspicious_pattern,
        regex=True,
        na=False
    )
)

print(
    "Potentially suspicious source images:"
)

print(
    cross_review[
        "suspicious_source"
    ].value_counts()
)

print(
    "\nSuspicious images:"
)

display(
    cross_review[
        cross_review["suspicious_source"]
    ][
        [
            "landmark",
            "filename",
            "title",
            "description_url"
        ]
    ].head(100)
)

Potentially suspicious source images:
suspicious_source
False    356
True      65
Name: count, dtype: int64

Suspicious images:


,landmark,filename,title,description_url
149,Alutila_Cave,0033_File_Small_Area_Atlas_of_Bangladesh_-_Kha...,File:Small Area Atlas of Bangladesh - Khagrach...,https://commons.wikimedia.org/wiki/File:Small_...
269,Anderson_Canal,0012_File_United_States_Civil_Aircraft_Registe...,File:United States Civil Aircraft Register Jan...,https://commons.wikimedia.org/wiki/File:United...
402,Banshbaria_Beach,0045_File_Small_Area_Atlas_of_Bangladesh_-_Bag...,File:Small Area Atlas of Bangladesh - Bagerhat...,https://commons.wikimedia.org/wiki/File:Small_...
557,Bayazid_Bostami_Shrine,0048_File_Journal_of_the_Bombay_Natural_Histor...,File:Journal of the Bombay Natural History Soc...,https://commons.wikimedia.org/wiki/File:Journa...
754,Central_Railway_Building,0097_File_The_New_York_Times_1930-04-28-_Vol_7...,File:The New York Times 1930-04-28- Vol 79 Iss...,https://commons.wikimedia.org/wiki/File:The_Ne...
...,...,...,...,...
5218,Three_Rivers_Mouth,0020_File_Board_of_Trade_Journal._London._1922...,File:Board of Trade Journal. London. 1922-06-0...,https://commons.wikimedia.org/wiki/File:Board_...
5223,Three_Rivers_Mouth,0025_File_Behar_Herald_8-12-2020_15-12-2020.pd...,File:Behar Herald 8-12-2020 & 15-12-2020.pdf,https://commons.wikimedia.org/wiki/File:Behar_...
5227,Three_Rivers_Mouth,0029_File_November_1961_-_DPLA_-_19f370327836f...,File:November 1961 - DPLA - 19f370327836f9576e...,https://commons.wikimedia.org/wiki/File:Novemb...
5237,Titas_River,0011_File_Small_Area_Atlas_of_Bangladesh_-_Bra...,File:Small Area Atlas of Bangladesh - Brahmanb...,https://commons.wikimedia.org/wiki/File:Small_...


In [ ]:
SOURCE_REVIEW_DIR = (
    "/content/drive/MyDrive/"
    "Chattogram_Landmark_Dataset/"
    "04_review/"
    "duplicate_analysis"
)

os.makedirs(SOURCE_REVIEW_DIR, exist_ok=True)

review_csv = os.path.join(
    SOURCE_REVIEW_DIR,
    "cross_class_contamination_review.csv"
)

cross_review.to_csv(
    review_csv,
    index=False
)

print("Saved:")
print(review_csv)

Saved:
/content/drive/MyDrive/Chattogram_Landmark_Dataset/04_review/duplicate_analysis/cross_class_contamination_review.csv
